# Brain Tumor MRI Classification: two datasets combined

Classify brain MRI slices into **glioma**, **meningioma**, **pituitary** tumor, or **no tumor**.

Training uses two public datasets together, so the model sees scans from more than one source:

| Dataset | Author(s) | Licence |
|---|---|---|
| [Brain Tumor Classification (MRI)](https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri) (Kaggle) | Bhuvaji, Kadam, Bhumkar, Dedge, Kanchan (2020) | MIT |
| [BDNeuro-MRI, Mendeley Data (Version 9)](https://data.mendeley.com/datasets/zwr4ntf94j) | Hira, Hossain, Bithee, Sara, Hasan, Al Towsif, Ahmed (2026) | CC BY 4.0 |

**What this notebook does differently from a single-dataset run**
1. Each image gets the same cleanup in training and at prediction time: grayscale, crop the black border, resize with padding, stretch contrast.
2. Duplicate and near-identical images are found across *both* datasets. Anything that appears in train, validation, or test more than once is kept only in the earliest of those splits, so no test image has a copy in training.
3. Each dataset keeps its own official test set. Results are reported per dataset and combined.
4. Model: DenseNet121 (ImageNet weights), trained in two stages, with class weights.

> Turn on a GPU first: **Runtime → Change runtime type → T4 GPU**.

## 1. Setup

In [ ]:
!pip install -q kagglehub

import os, glob, json, zipfile, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)

IMG_SIZE = 224
BATCH = 32
CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]
WEIGHTS = "imagenet"
EPOCHS_HEAD = 8      # stage 1: train the new classification head only
EPOCHS_FT = 30       # stage 2: fine-tune the whole network (early stopping ends it sooner)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Get both datasets

**Kaggle:** downloaded automatically (no API key needed, the dataset is public).

**Mendeley:** download the `.zip` from the [dataset page](https://data.mendeley.com/datasets/zwr4ntf94j) (use the latest version, 9) and put it in Colab either through Google Drive or by dragging it into the Files panel. Then set `ZIP_PATH` below.

In [ ]:
# Only needed if the Mendeley zip is on Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Set this to where your Mendeley zip is
ZIP_PATH = "/content/drive/MyDrive/Brain_Tumor_MRI_Dataset_Final.zip"
# ZIP_PATH = "/content/Brain_Tumor_MRI_Dataset_Final.zip"   # if uploaded directly

In [ ]:
import kagglehub

# Kaggle dataset
kaggle_path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")

# Mendeley dataset
EXTRACT_DIR = "/content/mendeley"
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
for inner in glob.glob(EXTRACT_DIR + "/**/*.zip", recursive=True):   # zip inside the zip
    with zipfile.ZipFile(inner) as z:
        z.extractall(os.path.dirname(inner))

print("Kaggle:  ", kaggle_path)
print("Mendeley:", EXTRACT_DIR)

## 3. Build one table of all images

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png")

def find_split(root, names):
    """Find a folder (e.g. 'train') that holds class subfolders full of images."""
    for r, dirs, _ in os.walk(root):
        if "__MACOSX" in r:
            continue
        for d in dirs:
            if d.lower() in names:
                p = os.path.join(r, d)
                subs = [s for s in os.listdir(p) if os.path.isdir(os.path.join(p, s))]
                if subs and any(f.lower().endswith(IMG_EXT)
                                for f in os.listdir(os.path.join(p, subs[0]))):
                    return p
    return None

def canonical(folder):
    """Map each dataset's folder names onto the same four class names."""
    n = folder.lower().replace("-", "_").replace(" ", "_")
    if n in ("no_tumor", "notumor", "normal", "healthy"):
        return "no_tumor"
    for c in ("glioma", "meningioma", "pituitary"):
        if n.startswith(c):
            return c
    raise ValueError(f"Unknown class folder: {folder}")

rows = []
def add_split(root, source, split):
    if root is None:
        return
    for cls in sorted(os.listdir(root)):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        label = canonical(cls)
        for f in os.listdir(cls_dir):
            if f.lower().endswith(IMG_EXT):
                rows.append(dict(path=os.path.join(cls_dir, f), label=label,
                                 source=source, split=split))

# Kaggle: Training / Testing
add_split(find_split(kaggle_path, {"training", "train"}), "kaggle", "train")
add_split(find_split(kaggle_path, {"testing", "test"}),   "kaggle", "test")

# Mendeley: train / val / test
add_split(find_split(EXTRACT_DIR, {"train", "training"}),            "mendeley", "train")
add_split(find_split(EXTRACT_DIR, {"val", "valid", "validation"}),   "mendeley", "val")
add_split(find_split(EXTRACT_DIR, {"test", "testing"}),              "mendeley", "test")

df = pd.DataFrame(rows)
print("Images found (before cleaning):", len(df))
print(pd.crosstab([df.source, df.split], df.label))

## 4. Clean the data: find duplicates and prevent leakage

Each image gets a small fingerprint (a 256-bit "difference hash" of a shrunken grayscale copy). Two images whose fingerprints differ in only a few bits are treated as the same picture, even if one was re-saved or slightly resized. This works within a dataset and *between* the two datasets.

Rules:
- A dataset with no validation folder gets 15% of its training images as validation.
- If the same picture appears with two different labels, both copies are dropped (the label can't be trusted).
- If a picture appears more than once, only the copy in the earliest split is kept (train, then validation, then test). So **no test image has a copy in train or validation**.

This only catches near-identical copies, not different slices from the same patient. `HASH_DIST` controls how strict the match is, and the cell after this one shows example matches so you can check it.

In [ ]:
HASH_DIST = 10   # fingerprints that differ in this many bits (of 256) or fewer = same picture

def fingerprint(path, size=16):
    img = Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(img, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

# 1) datasets with no validation split: carve 15% out of train
for src in df.source.unique():
    if not ((df.source == src) & (df.split == "val")).any():
        idx = df[(df.source == src) & (df.split == "train")].index
        _, val_idx = train_test_split(idx, test_size=0.15, random_state=SEED,
                                      stratify=df.loc[idx, "label"])
        df.loc[val_idx, "split"] = "val"

# 2) order rows: train first, then val, then test
df["prio"] = df.split.map({"train": 0, "val": 1, "test": 2})
df = df.sort_values("prio", kind="stable").reset_index(drop=True)
before = df.split.value_counts()
all_paths = df.path.values.copy()

# 3) compare every image with every earlier one
bits = np.stack([fingerprint(p) for p in tqdm(df.path, desc="Fingerprinting")])
X = bits.astype(np.float32) * 2 - 1
labels = df.label.values
n = len(df)
dup = np.zeros(n, dtype=bool)        # has an earlier near-identical copy
conflict = np.zeros(n, dtype=bool)   # near-identical copy with a different label
pairs = []                           # (later image, earlier image, distance)

for s in range(0, n, 500):
    e = min(s + 500, n)
    dist = (bits.shape[1] - X[s:e] @ X[:e].T) / 2
    earlier = np.arange(e)[None, :] < np.arange(s, e)[:, None]
    ii, jj = np.nonzero((dist <= HASH_DIST) & earlier)
    pairs += [(int(i) + s, int(j), int(dist[i, j])) for i, j in zip(ii, jj)]
    ii = ii + s
    dup[ii] = True
    differ = labels[ii] != labels[jj]
    conflict[ii[differ]] = True
    conflict[jj[differ]] = True

print(f"Near-identical pairs found: {len(pairs)} (exact fingerprint matches: {sum(p[2] == 0 for p in pairs)})")
print(f"Images dropped because a copy has a different label: {int(conflict.sum())}")

df = df[~(dup | conflict)].reset_index(drop=True)
after = df.split.value_counts()

print("\nRemoved as duplicates / leakage:")
for sp in ("train", "val", "test"):
    print(f"  {sp:5s} {before[sp]:5d} -> {after[sp]:5d}   (removed {before[sp] - after[sp]})")

df["label_id"] = df.label.map({c: i for i, c in enumerate(CLASS_NAMES)})
train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)

print("\nFinal counts:")
print(pd.crosstab([df.source, df.split], df.label, margins=True))

In [ ]:
# Sanity check: matched pairs that were not exact fingerprint matches
# (left = copy that was kept, right = copy that was removed). If these look like
# genuinely different slices, lower HASH_DIST and re-run from section 3.
inexact = [p for p in pairs if p[2] > 0]
print(f"{len(inexact)} matched pairs were close but not exact")
if inexact:
    shown = random.Random(SEED).sample(inexact, min(4, len(inexact)))
    fig, axes = plt.subplots(len(shown), 2, figsize=(6, 3 * len(shown)), squeeze=False)
    for k, (i, j, d) in enumerate(shown):
        for m, idx in enumerate((j, i)):
            axes[k][m].imshow(Image.open(all_paths[idx]).convert("L"), cmap="gray")
            axes[k][m].axis("off")
        axes[k][0].set_title(f"distance {d}", fontsize=9)
    plt.tight_layout(); plt.show()

## 5. Image preprocessing

One function, used for every image in training, evaluation, and your own uploads:
1. Read as grayscale (some sources are colour-mapped).
2. Crop away the black border around the head.
3. Resize to 224 x 224, padding with black so nothing is stretched or cut off.
4. Stretch contrast to the full 0 to 255 range.
5. Copy to 3 channels (the pretrained network expects colour input).

In [ ]:
@tf.autograph.experimental.do_not_convert
def crop_to_brain(img, thresh=20.0, min_frac=0.3):
    """Crop the dark border. Keeps the full image if the crop would be tiny."""
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]
    mask = img[:, :, 0] > thresh
    rows = tf.where(tf.reduce_any(mask, axis=1))[:, 0]
    cols = tf.where(tf.reduce_any(mask, axis=0))[:, 0]

    def crop():
        y0 = tf.cast(tf.reduce_min(rows), tf.int32)
        y1 = tf.cast(tf.reduce_max(rows), tf.int32) + 1
        x0 = tf.cast(tf.reduce_min(cols), tf.int32)
        x1 = tf.cast(tf.reduce_max(cols), tf.int32) + 1
        big_enough = tf.logical_and(
            tf.cast(y1 - y0, tf.float32) > min_frac * tf.cast(h, tf.float32),
            tf.cast(x1 - x0, tf.float32) > min_frac * tf.cast(w, tf.float32))
        return tf.cond(big_enough, lambda: img[y0:y1, x0:x1, :], lambda: img)

    return tf.cond(tf.size(rows) > 0, crop, lambda: img)

@tf.autograph.experimental.do_not_convert
def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=1, expand_animations=False)
    img.set_shape([None, None, 1])
    img = tf.cast(img, tf.float32)
    img = crop_to_brain(img)
    img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE)
    lo, hi = tf.reduce_min(img), tf.reduce_max(img)
    img = (img - lo) / (hi - lo + 1e-6) * 255.0
    img = tf.image.grayscale_to_rgb(img)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img

# Raw vs processed
sample = train_df.sample(4, random_state=1)
plt.figure(figsize=(12, 6))
for i, (_, r) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(r.path).convert("L"), cmap="gray")
    plt.title(f"{r.source}: raw", fontsize=9); plt.axis("off")
    plt.subplot(2, 4, i + 5)
    plt.imshow(load_image(r.path).numpy().astype("uint8"))
    plt.title(f"processed ({r.label})", fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()

## 6. Build the input pipelines

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

@tf.autograph.experimental.do_not_convert
def load_with_label(path, label):
    return load_image(path), label

def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label_id.values))
    ds = ds.map(load_with_label, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()                                  # decode each image only once
    if training:
        ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df)

# Class weights: rarer classes count more
weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)),
                               y=train_df.label_id.values)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print("Class weights:", {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

## 7. Build the model

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
])

base = keras.applications.DenseNet121(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=WEIGHTS)
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = keras.applications.densenet.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

## 8. Train

**Stage 1** trains only the new head while the pretrained network is frozen.
**Stage 2** unfreezes everything and trains slowly (learning rate 1e-5). Early stopping keeps the weights from the epoch with the lowest validation loss.

In [ ]:
# Stage 1: head only
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
es1 = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                     class_weight=class_weight, callbacks=[es1])

In [ ]:
# Stage 2: fine-tune the whole network
base.trainable = True
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
es2 = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
lr_drop = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                     class_weight=class_weight, callbacks=[es2, lr_drop])

In [ ]:
# Training curves, both stages joined
acc  = history1.history["accuracy"] + history2.history["accuracy"]
vacc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss = history1.history["loss"] + history2.history["loss"]
vloss = history1.history["val_loss"] + history2.history["val_loss"]
split_at = len(history1.history["loss"]) - 0.5

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (tr, va, title) in zip(ax, [(acc, vacc, "Accuracy"), (loss, vloss, "Loss")]):
    a.plot(tr, label="train"); a.plot(va, label="val")
    a.axvline(split_at, color="gray", linestyle="--", label="stage 2 starts")
    a.set_title(title); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 9. Evaluate

Test results are reported for each dataset's own test set and for both together. Test images that also appear in training were removed in section 4.

In [ ]:
def evaluate(frame, name, save_as=None):
    ds = make_ds(frame)
    probs = model.predict(ds, verbose=0)
    y_true = frame.label_id.values            # same order: the dataset is not shuffled
    y_pred = probs.argmax(axis=1)
    acc = float((y_true == y_pred).mean())

    print(f"\n===== {name}: {len(frame)} images, accuracy {acc:.3f} =====")
    print(classification_report(y_true, y_pred, labels=range(len(CLASS_NAMES)),
                                target_names=CLASS_NAMES, digits=3, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix: {name}")
    if save_as:
        plt.savefig(save_as, dpi=150, bbox_inches="tight")
    plt.show()
    return acc

results = {}
results["train"] = float(model.evaluate(make_ds(train_df), verbose=0)[1])
results["val"]   = float(model.evaluate(val_ds, verbose=0)[1])

for src in sorted(test_df.source.unique()):
    results[f"test ({src})"] = evaluate(test_df[test_df.source == src], f"{src} test set",
                                        save_as=f"confusion_matrix_{src}.png")
results["test (combined)"] = evaluate(test_df, "combined test set", save_as="confusion_matrix.png")

print("\nSummary")
for k, v in results.items():
    print(f"  {k:18s} {v:.1%}")

## 10. Look at some test predictions

In [ ]:
sample = test_df.sample(8, random_state=3).reset_index(drop=True)
imgs = np.stack([load_image(p).numpy() for p in sample.path])
probs = model.predict(imgs, verbose=0)

plt.figure(figsize=(12, 7))
for i in range(len(sample)):
    pred = CLASS_NAMES[probs[i].argmax()]
    true = sample.label[i]
    plt.subplot(2, 4, i + 1)
    plt.imshow(imgs[i].astype("uint8"))
    plt.title(f"pred: {pred}\ntrue: {true}\n({sample.source[i]})",
              color="green" if pred == true else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout(); plt.show()

## 11. Test your own image

Run the cell and pick one or more MRI images. They go through the same preprocessing as the training data.

The model always answers with one of the four classes, even for a picture that is not a brain MRI, so look at the confidence. Images from scanners or sequences the model has not seen may be classified poorly.

In [ ]:
from google.colab import files

uploaded = files.upload()

plt.figure(figsize=(8 * len(uploaded), 4))
for i, name in enumerate(uploaded):
    img = load_image(name)
    probs = model.predict(img.numpy()[None, ...], verbose=0)[0]
    pred = CLASS_NAMES[int(probs.argmax())]

    plt.subplot(1, 2 * len(uploaded), 2 * i + 1)
    plt.imshow(Image.open(name).convert("L"), cmap="gray"); plt.axis("off"); plt.title("your image")
    plt.subplot(1, 2 * len(uploaded), 2 * i + 2)
    plt.imshow(img.numpy().astype("uint8")); plt.axis("off")
    plt.title(f"{pred} ({probs.max():.0%})")

    print(name)
    for c, p in zip(CLASS_NAMES, probs):
        print(f"  {c:12s} {p:.1%}")
    if probs.max() < 0.6:
        print("  Low confidence: treat this prediction with suspicion.")
plt.tight_layout(); plt.show()

## 12. Save the model

In [ ]:
model.save("brain_tumor_model.keras")
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
print("Saved brain_tumor_model.keras and class_names.json")